In [5]:
from dotenv import load_dotenv, find_dotenv

_ = load_dotenv(find_dotenv())

In [ ]:
# Demo
from langchain.agents import create_agent

def get_weather(city: str) -> str:
    """Get weather for a given city."""
    return f"It's always sunny in {city}!"

agent = create_agent(
    model="openai:gpt-5.4",
    tools=[get_weather],
    system_prompt="You are a helpful assistant",
)

result = agent.invoke(
    {"messages": [{"role": "user", "content": "What's the weather in San Francisco?"}]}
)
print(result["messages"][-1].content_blocks)

[{'type': 'text', 'text': "It's always sunny in San Francisco!"}]


In [9]:
SYSTEM_PROMPT = """You are a literary data assistant.

## Capabilities

- `fetch_text_from_url`: loads document text from a URL into the conversation.
Do not guess line counts or positions—ground them in tool results from the saved file."""

In [3]:
import urllib.error
import urllib.request

from langchain.tools import tool

@tool
def fetch_text_from_url(url : str) -> str:
    """Fetch the document from a URL.
    """
    req = urllib.request.Request(
        url=url,
        headers={"User-Agent": "Mozilla/5.0 (compatible; quickstart-research/1.0)"},
    )

    try: 
        with urllib.request.urlopen(req) as resp:
            raw = resp.read()
    except urllib.error.URLError as e:
        return f"Fetch failed: {e}"
    
    text = raw.decode("utf-8", errors="replace")
    return text


In [6]:
from langchain.chat_models import init_chat_model

model = init_chat_model(
    "openai:gpt-5.4",
    temperature=0.5,
    timeout=300,
    max_tokens=25000,
)

In [7]:
from langgraph.checkpoint.memory import InMemorySaver

checkpointer = InMemorySaver()

In [10]:
from langchain.agents import create_agent
from deepagents import create_deep_agent

agent = create_agent(
    model=model,
    tools=[fetch_text_from_url],
    system_prompt=SYSTEM_PROMPT,
    checkpointer=checkpointer,
)

deep_agent = create_deep_agent(
    model=model,
    tools=[fetch_text_from_url],
    system_prompt=SYSTEM_PROMPT,
    checkpointer=checkpointer,
)

content = f"""Project Gutenberg hosts a full plain-text copy of F. Scott Fitzgerald's The Great Gatsby.
URL: https://www.gutenberg.org/files/64317/64317-0.txt

Answer as much as you can:

1) How many lines in the complete Gutenberg file contain the substring `Gatsby` (count lines, not occurrences within a line, each line ends with a line break).
2) The 1-based line number of the first line in the file that contains `Daisy`.
3) A two-sentence neutral synopsis.

Do your best on (1) and (2). If at any point you realize you cannot **verify** an exact answer with
your available tools and reasoning, do not fabricate numbers: use `null` for that field and spell out
the limitation in `how_you_computed_counts`. If you encounter any errors please report what the error was and what the error message was."""

agent_result = agent.invoke(
    {"messages": [{"role": "user", "content": content}]},
    config={"configurable": {"thread_id": "great-gatsby-lc"}},
)
deep_agent_result = deep_agent.invoke(
    {"messages": [{"role": "user", "content": content}]},
    config={"configurable": {"thread_id": "great-gatsby-da"}},
)
print(agent_result["messages"][-1].content_blocks)
print("\n")
print(deep_agent_result["messages"][-1].content_blocks)

[{'type': 'text', 'text': '{\n  "gatsby_line_count": null,\n  "first_daisy_line_number": null,\n  "synopsis": "The Great Gatsby follows narrator Nick Carraway as he becomes drawn into the world of the wealthy Jay Gatsby and the social circle surrounding Daisy Buchanan on Long Island in the summer of 1922. Through Gatsby\'s pursuit of Daisy and the conflicts among the characters, the novel depicts aspiration, class tension, and disillusionment in the Jazz Age.",\n  "how_you_computed_counts": "I attempted to load the Project Gutenberg plain-text file directly from the provided URL using the available document-fetching tool so I could count lines containing the substring \'Gatsby\' and identify the first 1-based line containing \'Daisy\'. The fetch failed, so I could not verify exact line-based answers from the file; per your instruction, I am returning null for those fields rather than fabricating values.",\n  "errors": [\n    {\n      "step": "fetch_text_from_url",\n      "url": "https: